# RSnowflake Performance Enhancements -- Test & Benchmark

This notebook tests and benchmarks the performance improvements on the
`perf/rsnowflake-optimizations` branch.

**Architecture (confirmed by Snowflake engineering, Mar 2026):**
- **REST API v2** is the universal baseline -- JSON-only, no Arrow result format (gated/not GA), no array binding, no PUT/COPY/stage upload.
- **ADBC** (when installed) connects to the **public endpoint** as an external client using PAT auth. Works in Workspace with appropriate network/auth policy.
- **Session-based** (GS protocol) features are blocked in Workspace -- PAT/SPCS tokens are rejected by the internal login endpoint.

**Enhancements tested:**
1. Optimised JSON parsing (matrix transpose + RcppSimdJson)
2. Parallel partition fetching
3. Data upload methods (literal SQL INSERT vs. ADBC bulk ingest)
4. Session-based connection & transactions (expected to be unavailable in Workspace)
5. Native Arrow transport (requires session -- unavailable in Workspace)
6. ADBC backend acceleration (reads and writes via internal SPCS host)

**How benchmarks work:**
Each enhancement is toggled on/off via `options()` so we can compare the
old path against the new path using identical queries. Timings are collected
with `system.time()` and summarised in a final comparison table.

## 0. Setup

Single cell to set session context, validate EAI, install R, install RSnowflake,
and export SPCS OAuth env vars.

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="rsnowflake_perf_config.yaml", packages=["RSnowflake"])

In [ ]:
%%R
# Verify core dependencies (installed by setup script)
pkgs <- c("DBI", "httr2", "jsonlite", "rlang", "cli")
missing <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  cat("Installing missing core deps:", paste(missing, collapse = ", "), "\n")
  install.packages(missing, repos = "https://cloud.r-project.org")
}
cat("Core dependencies OK.\n")

# ADBC status (installed by setup script --adbc flag)
cat("adbcdrivermanager:", requireNamespace("adbcdrivermanager", quietly = TRUE), "\n")
cat("adbcsnowflake:    ", requireNamespace("adbcsnowflake", quietly = TRUE), "\n")
if (!requireNamespace("adbcsnowflake", quietly = TRUE)) {
  cat("NOTE: ADBC not installed. Re-run setup with --adbc for Arrow-native performance.\n")
}

### Connect via SPCS OAuth

In [ ]:
# SPCS OAuth env vars already exported by setup_notebook()
import os

print(f"Account:   {os.environ['SNOWFLAKE_ACCOUNT']}")
print(f"Database:  {os.environ['SNOWFLAKE_DATABASE']}")
print(f"Warehouse: {os.environ['SNOWFLAKE_WAREHOUSE']}")
print("Auth:      SPCS OAuth token (built-in)")

In [ ]:
%%R
if (!nzchar(Sys.getenv("TZ", ""))) Sys.setenv(TZ = "UTC")

library(RSnowflake)
library(DBI)

con <- dbConnect(Snowflake())
cat("Connected:", dbIsValid(con), "\n")

In [ ]:
%%R
# Helper: run a timed expression and return elapsed seconds.
# Uses substitute()/eval() so the expression is re-evaluated each iteration
# (force() only evaluates the promise once, making runs 2+ instant).
bench <- function(label, expr, times = 3L) {
  expr_q <- substitute(expr)
  env    <- parent.frame()
  timings <- numeric(times)
  result <- NULL
  for (i in seq_len(times)) {
    t0 <- proc.time()
    result <- eval(expr_q, env)
    timings[i] <- (proc.time() - t0)[["elapsed"]]
  }
  med <- median(timings)
  cat(sprintf("  %-45s  median: %6.3fs  [%s]\n",
              label, med,
              paste(sprintf("%.3f", timings), collapse = ", ")))
  list(label = label, median = med, timings = timings, result = result)
}

# Accumulator for the final comparison table
results <- list()
cat("Benchmark helper ready.\n")

## 1. Optimised JSON Parsing

**What changed:** The `.json_data_to_df()` hot path now uses a single-pass
matrix transpose instead of per-element `vapply`.  Additionally, when
`RcppSimdJson` is installed, the raw HTTP response is parsed with
`RcppSimdJson::fparse()` instead of `jsonlite` (3-5x faster for large JSON).

**Test:** Fetch 50K rows with SimdJson enabled vs. disabled.

In [ ]:
%%R
cat("RcppSimdJson available:",
    requireNamespace("RcppSimdJson", quietly = TRUE), "\n")

N <- 50000L
sql <- sprintf("
  SELECT
    SEQ4()              AS id,
    RANDSTR(20, RANDOM()) AS txt,
    UNIFORM(0::FLOAT, 100::FLOAT, RANDOM()) AS score,
    IFF(MOD(SEQ4(), 2) = 0, TRUE, FALSE) AS flag,
    DATEADD(DAY, MOD(SEQ4(), 365), '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => %d))
", N)

cat("\n--- SimdJson ON (default) ---\n")
options(RSnowflake.use_simdjson = TRUE)
r1 <- bench("50K rows, SimdJson ON",
            dbGetQuery(con, sql))
results[[length(results) + 1L]] <- r1

cat("\n--- SimdJson OFF (jsonlite fallback) ---\n")
options(RSnowflake.use_simdjson = FALSE)
r2 <- bench("50K rows, SimdJson OFF",
            dbGetQuery(con, sql))
results[[length(results) + 1L]] <- r2

# Restore default
options(RSnowflake.use_simdjson = TRUE)

cat(sprintf("\nSpeedup: %.1fx\n", r2$median / r1$median))

In [ ]:
%%R
# Correctness check: verify types from the SimdJson path
df <- r1$result
cat("Rows:", nrow(df), "  Cols:", ncol(df), "\n")
cat("ID type:   ", typeof(df$ID), "\n")
cat("TXT type:  ", typeof(df$TXT), "\n")
cat("SCORE type:", typeof(df$SCORE), "\n")
cat("FLAG type: ", typeof(df$FLAG), "\n")
cat("DT class:  ", class(df$DT), "\n")
stopifnot(nrow(df) == N)
stopifnot(is.integer(df$ID))
stopifnot(is.character(df$TXT))
stopifnot(is.double(df$SCORE))
stopifnot(is.logical(df$FLAG))
stopifnot(inherits(df$DT, "Date"))
cat("All type assertions passed.\n")

## 2. Parallel Partition Fetching

**What changed:** When a query result has multiple partitions (typically >100K
rows), remaining partitions are now fetched in parallel using `mclapply`
instead of sequentially. The worker count auto-detects via
`parallel::detectCores()`, adapting to the SPCS container's instance family
(e.g. CPU_X64_XS = 1 vCPU, CPU_X64_S = 3, CPU_X64_M = 6, etc.).

**Test:** Fetch 200K rows (likely multi-partition) with parallel ON vs. OFF.

In [ ]:
%%R
N_LARGE <- 200000L
sql_large <- sprintf("
  SELECT
    SEQ4()              AS id,
    RANDSTR(30, RANDOM()) AS txt,
    UNIFORM(0::FLOAT, 1000::FLOAT, RANDOM()) AS val
  FROM TABLE(GENERATOR(ROWCOUNT => %d))
", N_LARGE)

# Show auto-detected core count for this container
detected <- parallel::detectCores(logical = FALSE)
cat(sprintf("Container vCPUs (physical): %s\n", detected))
cat(sprintf("Auto-detected workers: %d\n\n",
            max(1L, detected - 1L)))

cat("--- Parallel fetch ON (auto-detect workers) ---\n")
options(RSnowflake.parallel_fetch = TRUE, RSnowflake.fetch_workers = 0L)
r3 <- bench(sprintf("200K rows, parallel ON (%d workers)", max(1L, detected - 1L)),
            dbGetQuery(con, sql_large))
results[[length(results) + 1L]] <- r3

cat("\n--- Parallel fetch OFF (sequential) ---\n")
options(RSnowflake.parallel_fetch = FALSE)
r4 <- bench("200K rows, parallel OFF",
            dbGetQuery(con, sql_large))
results[[length(results) + 1L]] <- r4

# Restore
options(RSnowflake.parallel_fetch = TRUE, RSnowflake.fetch_workers = 0L)

cat(sprintf("\nRows returned: %d (parallel) vs %d (sequential)\n",
            nrow(r3$result), nrow(r4$result)))
cat(sprintf("Speedup: %.1fx\n", r4$median / r3$median))

In [ ]:
%%R
# Correctness: both paths should return the same shape
stopifnot(nrow(r3$result) == N_LARGE)
stopifnot(nrow(r4$result) == N_LARGE)
stopifnot(ncol(r3$result) == ncol(r4$result))
cat("Parallel fetch correctness OK.\n")

## 3. Data Upload Methods

**What changed:** `dbWriteTable()` routing (default `upload_method = "auto"`):
- For large data (cells >= `adbc_write_threshold`), uses **ADBC bulk ingest**
  (PUT + COPY INTO via the Go driver) when ADBC is installed.
- For small data or when ADBC is unavailable, uses **SQL-literal INSERT**
  (the fastest REST-only path).
- The bind-parameter path (`"bind"`) is **deprecated** -- Snowflake engineering
  confirmed that SQL API v2 has no array-binding support, making row-wise
  binds significantly slower than literal INSERT.

**Workspace note (updated Mar 2026):** ADBC writes now route through the
**internal SPCS gateway** (`SNOWFLAKE_HOST`) using SPCS OAuth. This eliminates
the previous overhead of the public endpoint + PAT auth pattern. ADBC writes
excel for **large datasets** (50K+ rows) where bulk throughput (PUT + COPY INTO)
outweighs the fixed overhead. Stage hosts from `SYSTEM$ALLOWLIST()` may no
longer be needed (under investigation).

**Test:** Upload 5K rows via literal (REST) vs. bind-parameter INSERT (deprecated).

In [ ]:
%%R
# Create test data
set.seed(42)
upload_n <- 5000L
upload_df <- data.frame(
  id     = seq_len(upload_n),
  name   = paste0("item_", seq_len(upload_n)),
  value  = runif(upload_n, 0, 1000),
  active = sample(c(TRUE, FALSE), upload_n, replace = TRUE),
  stringsAsFactors = FALSE
)
cat(sprintf("Test data: %d rows x %d cols\n", nrow(upload_df), ncol(upload_df)))

# Ensure a clean test schema
tryCatch(dbExecute(con, "CREATE SCHEMA IF NOT EXISTS PERF_TEST"), error = function(e) NULL)

cat("\n--- SQL-literal INSERT (default) ---\n")
options(RSnowflake.upload_method = "literal")
r5 <- bench(sprintf("%dK rows, literal INSERT", upload_n / 1000L), {
  dbWriteTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_LIT"),
               upload_df, overwrite = TRUE)
})
results[[length(results) + 1L]] <- r5

cat("\n--- Bind-parameter INSERT ---\n")
options(RSnowflake.upload_method = "bind")
r6 <- bench(sprintf("%dK rows, bind INSERT", upload_n / 1000L), {
  dbWriteTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_BIND"),
               upload_df, overwrite = TRUE)
})
results[[length(results) + 1L]] <- r6

# Restore default
options(RSnowflake.upload_method = "literal")

cat(sprintf("\nLiteral vs bind: %.1fx faster\n", r6$median / r5$median))

In [ ]:
%%R
# Correctness: read back and verify
bind_back <- dbReadTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_BIND"))
lit_back  <- dbReadTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_LIT"))

cat("Bind rows read back: ", nrow(bind_back), "\n")
cat("Literal rows read back:", nrow(lit_back), "\n")
stopifnot(nrow(bind_back) == upload_n)
stopifnot(nrow(lit_back) == upload_n)
cat("Upload correctness OK.\n")

In [ ]:
%%R
# Larger upload test: 20K rows (literal default)
upload_n2 <- 20000L
upload_df2 <- data.frame(
  id    = seq_len(upload_n2),
  txt   = paste0("row_", seq_len(upload_n2), "_", strrep("x", 50)),
  val   = rnorm(upload_n2),
  stringsAsFactors = FALSE
)

cat("--- 20K rows, literal INSERT ---\n")
options(RSnowflake.upload_method = "literal")
r7 <- bench("20K rows, literal INSERT", {
  dbWriteTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_LIT_20K"),
               upload_df2, overwrite = TRUE)
})
results[[length(results) + 1L]] <- r7

cnt <- dbGetQuery(con, "SELECT COUNT(*) AS n FROM PERF_TEST.BENCH_LIT_20K")
cat("Rows in table:", cnt$N, "\n")
stopifnot(cnt$N == upload_n2)
cat("20K literal upload OK.\n")

## 4. Session-Based Connection & Transactions

**What changed:** When `options(RSnowflake.use_session = TRUE)` is set
before connecting, RSnowflake establishes a persistent session via
`/session/v1/login-request`. This enables:
- `dbBegin()` / `dbCommit()` / `dbRollback()` / `dbWithTransaction()`
- Temporary tables that persist across requests
- Session variables

**Workspace limitation (confirmed by Snowflake engineering, Mar 2026):**
The internal GS login endpoint (`/session/v1/login-request`) expects
OAuth/driver flows, not PAT. There is **no supported way** for a custom
R connector inside a Workspace Notebook to open a GS session using either
the PAT from `session.create_pat()` or the SPCS token. This section will
show session login failing -- which is the expected behaviour.

**Test:** Attempt session login, then test transaction support.

In [ ]:
%%R
# First, verify transactions fail on stateless connection (expected)
tryCatch({
  dbBegin(con)
  cat("ERROR: dbBegin should have failed on stateless connection!\n")
}, error = function(e) {
  cat("Stateless transaction test: PASS\n")
  cat("  Error:", conditionMessage(e), "\n")
})

In [ ]:
%%R
# Attempt a session-based connection
options(RSnowflake.use_session = TRUE)
session_con <- tryCatch({
  dbConnect(Snowflake())
}, error = function(e) {
  cat("Session connection failed:", conditionMessage(e), "\n")
  NULL
})

if (!is.null(session_con) && dbIsValid(session_con)) {
  has_session <- !is.null(session_con@.state$session)
  cat("Session-based:", has_session, "\n")
  if (has_session) {
    cat("Session ID:", session_con@.state$session$session_id, "\n")
  } else {
    cat("Fell back to stateless SQL API v2 (session login failed).\n")
    cat("This is expected if the internal login endpoint is blocked.\n")
  }
} else {
  cat("Could not establish any connection with use_session=TRUE.\n")
  cat("Falling back to stateless connection for remaining tests.\n")
  session_con <- con
}
options(RSnowflake.use_session = FALSE)

In [ ]:
%%R
# Test transactions if session is active
if (!is.null(session_con@.state$session)) {
  cat("--- Transaction tests ---\n")

  # Test 1: BEGIN / COMMIT
  tryCatch({
    dbBegin(session_con)
    dbExecute(session_con, "CREATE OR REPLACE TEMP TABLE PERF_TEST.TXN_TEST (x INT)")
    dbExecute(session_con, "INSERT INTO PERF_TEST.TXN_TEST VALUES (1), (2), (3)")
    dbCommit(session_con)
    cnt <- dbGetQuery(session_con, "SELECT COUNT(*) AS n FROM PERF_TEST.TXN_TEST")
    cat("BEGIN/COMMIT test: PASS  (rows =", cnt$N, ")\n")
  }, error = function(e) {
    cat("BEGIN/COMMIT test: FAIL -", conditionMessage(e), "\n")
  })

  # Test 2: BEGIN / ROLLBACK
  tryCatch({
    dbBegin(session_con)
    dbExecute(session_con, "INSERT INTO PERF_TEST.TXN_TEST VALUES (99)")
    dbRollback(session_con)
    cnt2 <- dbGetQuery(session_con, "SELECT COUNT(*) AS n FROM PERF_TEST.TXN_TEST")
    if (cnt2$N == 3L) {
      cat("BEGIN/ROLLBACK test: PASS  (rows still =", cnt2$N, ")\n")
    } else {
      cat("BEGIN/ROLLBACK test: FAIL  (expected 3, got", cnt2$N, ")\n")
    }
  }, error = function(e) {
    cat("BEGIN/ROLLBACK test: FAIL -", conditionMessage(e), "\n")
  })

  # Test 3: dbWithTransaction
  tryCatch({
    val <- dbWithTransaction(session_con, {
      dbExecute(session_con, "INSERT INTO PERF_TEST.TXN_TEST VALUES (4)")
      42
    })
    cat("dbWithTransaction test: PASS  (returned", val, ")\n")
  }, error = function(e) {
    cat("dbWithTransaction test: FAIL -", conditionMessage(e), "\n")
  })
} else {
  cat("Skipping transaction tests (no active session).\n")
  cat("This is expected -- the internal login endpoint may not be\n")
  cat("accessible from Workspace Notebooks.\n")
}

## 5. Native Arrow Transport (GS Session Required)

**What changed:** When a session is active and
`options(RSnowflake.use_native_arrow = TRUE)`, `dbGetQueryArrow()` submits
queries via the internal protocol requesting Arrow IPC format.

**Workspace limitation:** Requires a GS session (Section 4), which is
**not available** in Workspace Notebooks. This section will be skipped.
For Arrow-native reads in Workspace, use **ADBC** (Section 6) instead --
ADBC provides true server-side Arrow transport via the Go driver without
needing a GS session.

**Note:** SQL API v2 does not support Arrow result format (the internal
`ENABLE_SQL_API_ARROW_V1` parameter is not GA and subject to change).
The client-side Arrow path (JSON -> nanoarrow conversion) is always available.

**Test:** Fetch 50K rows via native Arrow vs. client-side Arrow conversion.

In [ ]:
%%R
has_nanoarrow <- requireNamespace("nanoarrow", quietly = TRUE)
has_active_session <- !is.null(session_con@.state$session)

cat("nanoarrow available: ", has_nanoarrow, "\n")
cat("Active session:      ", has_active_session, "\n")

if (has_nanoarrow && has_active_session) {
  sql_arrow <- sprintf("
    SELECT SEQ4() AS id, RANDSTR(20, RANDOM()) AS txt,
           UNIFORM(0::FLOAT, 100::FLOAT, RANDOM()) AS val
    FROM TABLE(GENERATOR(ROWCOUNT => %d))
  ", N)

  cat("\n--- Native Arrow transport ---\n")
  options(RSnowflake.use_native_arrow = TRUE)
  r8 <- bench("50K rows, native Arrow", {
    stream <- dbGetQueryArrow(session_con, sql_arrow)
    as.data.frame(stream)
  })
  results[[length(results) + 1L]] <- r8

  cat("\n--- Client-side Arrow (JSON + nanoarrow conversion) ---\n")
  options(RSnowflake.use_native_arrow = FALSE)
  r9 <- bench("50K rows, client-side Arrow", {
    stream <- dbGetQueryArrow(session_con, sql_arrow)
    as.data.frame(stream)
  })
  results[[length(results) + 1L]] <- r9

  options(RSnowflake.use_native_arrow = FALSE)

  cat(sprintf("\nNative Arrow rows: %d\n", nrow(r8$result)))
  cat(sprintf("Speedup: %.1fx\n", r9$median / r8$median))

} else if (has_nanoarrow) {
  cat("\nSkipping native Arrow test (no active session).\n")
  cat("Testing client-side Arrow path only.\n\n")

  sql_arrow <- sprintf("
    SELECT SEQ4() AS id, RANDSTR(20, RANDOM()) AS txt,
           UNIFORM(0::FLOAT, 100::FLOAT, RANDOM()) AS val
    FROM TABLE(GENERATOR(ROWCOUNT => %d))
  ", N)

  r9 <- bench("50K rows, client-side Arrow", {
    stream <- dbGetQueryArrow(con, sql_arrow)
    as.data.frame(stream)
  })
  results[[length(results) + 1L]] <- r9

  cat("Client-side Arrow rows:", nrow(r9$result), "\n")
} else {
  cat("Skipping Arrow tests (nanoarrow not installed).\n")
}

## 6. ADBC Backend Acceleration

**What changed:** When `adbcsnowflake` and `adbcdrivermanager` are installed,
RSnowflake can transparently route reads and writes through the Snowflake ADBC
Go driver. ADBC uses Arrow-native transport for reads and PUT + COPY INTO for
writes, bypassing the REST API's JSON serialisation overhead.

**Workspace note (updated Mar 2026):** ADBC now connects via the **internal
SPCS gateway** (`SNOWFLAKE_HOST`) using the SPCS OAuth token with `auth_oauth`.
No PAT, public endpoint, or authentication policy configuration is needed.
This should improve performance compared to the previous PAT + public endpoint
pattern by eliminating the external network round-trip. Prior benchmarks showed
ADBC reads at ~25s (init) / ~21s (steady) via the public endpoint -- compare
with the results below to validate the improvement.

- `options(RSnowflake.backend = "auto")` (default) uses ADBC when available
- `options(RSnowflake.backend = "rest")` forces the REST API v2 path
- `options(RSnowflake.backend = "adbc")` forces ADBC (falls back to REST on error)
- `options(RSnowflake.upload_method = "auto")` uses ADBC for large writes, literal for small

In [ ]:
%%R
has_adbc <- requireNamespace("adbcsnowflake", quietly = TRUE) &&
            requireNamespace("adbcdrivermanager", quietly = TRUE)

cat("adbcsnowflake installed:", has_adbc, "\n")

if (has_adbc) {
  # Warm up ADBC backend (first call includes Go driver init + TLS + PAT auth)
  options(RSnowflake.backend = "adbc")
  t_init <- proc.time()
  warmup <- dbGetQuery(con, "SELECT 1 AS x")
  init_elapsed <- (proc.time() - t_init)[["elapsed"]]
  cat(sprintf("ADBC init: %.1fs (one-time cost)\n", init_elapsed))

  sql_adbc <- sprintf("
    SELECT SEQ4() AS id, RANDSTR(20, RANDOM()) AS txt,
           UNIFORM(0::FLOAT, 100::FLOAT, RANDOM()) AS score,
           IFF(MOD(SEQ4(), 2) = 0, TRUE, FALSE) AS flag,
           DATEADD(DAY, MOD(SEQ4(), 365), '2024-01-01'::DATE) AS dt
    FROM TABLE(GENERATOR(ROWCOUNT => %d))
  ", N)

  cat("\n--- ADBC read (Arrow-native, steady-state) ---\n")
  r10 <- bench("50K rows, ADBC read", dbGetQuery(con, sql_adbc))
  results[[length(results) + 1L]] <- r10

  cat("\n--- REST API read (JSON) ---\n")
  options(RSnowflake.backend = "rest")
  r11 <- bench("50K rows, REST read", dbGetQuery(con, sql_adbc))
  results[[length(results) + 1L]] <- r11

  options(RSnowflake.backend = "auto")

  cat(sprintf("\nADBC read rows: %d\n", nrow(r10$result)))
  cat(sprintf("REST read rows: %d\n", nrow(r11$result)))
  cat(sprintf("Speedup: %.1fx\n", r11$median / r10$median))
} else {
  cat("Skipping ADBC read benchmark (adbcsnowflake not installed).\n")
  cat("Install with: install.packages('adbcsnowflake')\n")
}

In [ ]:
%%R
if (has_adbc) {
  cat("--- ADBC bulk write (PUT + COPY INTO) ---\n")
  options(RSnowflake.upload_method = "adbc")
  r12 <- bench(sprintf("%dK rows, ADBC write", upload_n / 1000L), {
    dbWriteTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_ADBC"),
                 upload_df, overwrite = TRUE)
  })
  results[[length(results) + 1L]] <- r12

  cat("\n--- Literal INSERT (REST API) ---\n")
  options(RSnowflake.upload_method = "literal")
  r13 <- bench(sprintf("%dK rows, literal write", upload_n / 1000L), {
    dbWriteTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_LIT2"),
                 upload_df, overwrite = TRUE)
  })
  results[[length(results) + 1L]] <- r13

  # Snowpark write_pandas (if reticulate + snowpark available)
  has_snowpark <- tryCatch({
    requireNamespace("reticulate", quietly = TRUE) &&
      !is.null(reticulate::import("snowflake.snowpark", delay_load = FALSE))
  }, error = function(e) FALSE)

  if (has_snowpark) {
    cat("\n--- Snowpark write_pandas (via reticulate) ---\n")
    options(RSnowflake.upload_method = "snowpark")
    r14 <- bench(sprintf("%dK rows, Snowpark write", upload_n / 1000L), {
      dbWriteTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_SNOWPARK"),
                   upload_df, overwrite = TRUE)
    })
    results[[length(results) + 1L]] <- r14

    sp_back <- dbReadTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_SNOWPARK"))
    cat(sprintf("Snowpark round-trip rows: %d (expected %d)\n", nrow(sp_back), upload_n))
  }

  options(RSnowflake.upload_method = "auto")

  speedup <- r13$median / r12$median
  cat(sprintf("\nADBC vs literal speedup: %.1fx\n", speedup))
  if (has_snowpark) {
    cat(sprintf("Snowpark vs literal speedup: %.1fx\n", r13$median / r14$median))
  }

  adbc_back <- dbReadTable(con, DBI::Id(schema = "PERF_TEST", table = "BENCH_ADBC"))
  cat(sprintf("ADBC round-trip rows: %d (expected %d)\n", nrow(adbc_back), upload_n))

  if (nzchar(Sys.getenv("SNOWFLAKE_HOST", ""))) {
    if (has_snowpark) {
      cat("\nWorkspace: 'auto' mode routes bulk writes to Snowpark (internal SPCS path).\n")
      cat("ADBC routes through the public endpoint; use explicit 'adbc' to force it.\n")
    }
    if (speedup < 1.0) {
      cat("ADBC was slower than literal INSERT for this dataset size.\n")
      cat("ADBC excels at 50K+ rows. Check STAGE hosts from SYSTEM$ALLOWLIST().\n")
    }
  }
} else {
  cat("Skipping ADBC write benchmark (adbcsnowflake not installed).\n")
}

## 7. Performance Summary

Consolidated comparison of all benchmarks.

In [ ]:
%%R
cat("\n")
cat("==========================================================================\n")
cat("  RSnowflake Performance Benchmark Summary\n")
cat("==========================================================================\n")
cat(sprintf("  %-45s  %10s\n", "Test", "Median (s)"))
cat("--------------------------------------------------------------------------\n")
for (r in results) {
  cat(sprintf("  %-45s  %10.3f\n", r$label, r$median))
}
cat("==========================================================================\n")

# Compute speedups for paired tests
cat("\n")
cat("Speedups (new / old):\n")
cat("--------------------------------------------------------------------------\n")

pairs <- list(
  list(new = "50K rows, SimdJson ON",
       old = "50K rows, SimdJson OFF",
       label = "JSON Parsing (SimdJson)"),
  list(new = sprintf("200K rows, parallel ON (%d workers)", max(1L, detected - 1L)),
       old = "200K rows, parallel OFF",
       label = "Partition Fetch (parallel)"),
  list(new = "5K rows, literal INSERT",
       old = "5K rows, bind INSERT",
       label = "Data Upload (literal vs bind)"),
  list(new = "50K rows, native Arrow",
       old = "50K rows, client-side Arrow",
       label = "Arrow Transport (native)"),
  list(new = "50K rows, ADBC read",
       old = "50K rows, REST read",
       label = "ADBC Read (vs REST)"),
  list(new = sprintf("%dK rows, ADBC write", upload_n / 1000L),
       old = sprintf("%dK rows, literal write", upload_n / 1000L),
       label = "ADBC Write (vs literal)")
)

lookup <- setNames(
  vapply(results, function(r) r$median, numeric(1)),
  vapply(results, function(r) r$label, character(1))
)

for (p in pairs) {
  if (p$new %in% names(lookup) && p$old %in% names(lookup)) {
    speedup <- lookup[[p$old]] / lookup[[p$new]]
    cat(sprintf("  %-35s  %.1fx  (%.3fs -> %.3fs)\n",
                p$label, speedup, lookup[[p$old]], lookup[[p$new]]))
  }
}
cat("==========================================================================\n")

## 8. Cleanup

Drop test tables and disconnect.

In [ ]:
%%R
# Drop test tables
for (tbl in c("BENCH_BIND", "BENCH_LIT", "BENCH_LIT_20K", "BENCH_ADBC", "BENCH_LIT2", "BENCH_SNOWPARK", "TXN_TEST")) {
  tryCatch(
    dbExecute(con, sprintf("DROP TABLE IF EXISTS PERF_TEST.%s", tbl)),
    error = function(e) NULL
  )
}
tryCatch(dbExecute(con, "DROP SCHEMA IF EXISTS PERF_TEST"), error = function(e) NULL)
cat("Test tables cleaned up.\n")

# Disconnect
if (exists("session_con") && !identical(session_con, con) && dbIsValid(session_con)) {
  dbDisconnect(session_con)
  cat("Session connection disconnected.\n")
}
dbDisconnect(con)
cat("Main connection disconnected.\n")

## Current Options Reference

| Option | Default | Description |
| --- | --- | --- |
| `RSnowflake.backend` | `"auto"` | `"auto"` (ADBC if available, else REST), `"rest"`, or `"adbc"` |
| `RSnowflake.use_simdjson` | `TRUE` | Use RcppSimdJson for JSON parsing when installed |
| `RSnowflake.parallel_fetch` | `TRUE` | Fetch partitions in parallel |
| `RSnowflake.fetch_workers` | `0` (auto) | Parallel workers; 0 = auto-detect from container vCPUs |
| `RSnowflake.upload_method` | `"auto"` | `"auto"` (Workspace: Snowpark > ADBC > literal; external: ADBC > literal), `"snowpark"`, `"adbc"`, or `"literal"`. `"bind"` is **deprecated**. |
| `RSnowflake.bulk_write_threshold` | `50000` (local) / `200000` (Workspace) | Cell count (rows x cols) above which bulk write is used in `"auto"` mode. Workspace routes to Snowpark write_pandas; external routes to ADBC. |
| `RSnowflake.adbc_write_threshold` | (alias) | Backwards-compatible alias for `bulk_write_threshold`. |
| `RSnowflake.insert_batch_size` | `16384` | Rows per INSERT batch (literal path) |
| `RSnowflake.use_session` | `FALSE` | Establish a persistent GS session on connect (not available in Workspace) |
| `RSnowflake.use_native_arrow` | `FALSE` | Use native Arrow IPC transport (requires GS session -- not available in Workspace) |

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.PERF_TEST_RESULTS AS
        SELECT 'perf_test' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.PERF_TEST_RESULTS")
except Exception:
    pass